In [1]:
# Install Required Dependencies
print("Installing required dependencies...")

import subprocess
import sys

def install_package(package_name, description=""):
    """Install a package using pip"""
    try:
        print(f"Installing {package_name}... {description}")
        result = subprocess.run([sys.executable, "-m", "pip", "install", package_name], 
                              capture_output=True, text=True, check=True)
        print(f"✅ {package_name} installed successfully!")
        return True
    except subprocess.CalledProcessError as e:
        print(f"❌ Error installing {package_name}: {e}")
        print(f"Error output: {e.stderr}")
        return False

# Install SentencePiece (required for T5 tokenizer)
success = install_package("sentencepiece", "(Required for T5 tokenizer)")

if success:
    print("\n All dependencies installed successfully!")
    print("You can now proceed with the training.")
else:
    print("\n Installation failed. Please try manually:")
    print("pip install sentencepiece")

print("\nRestarting kernel is recommended after installing new packages...")
print("="*60)

Installing required dependencies...
Installing sentencepiece... (Required for T5 tokenizer)
❌ Error installing sentencepiece: Command '['d:\\padhai\\gaurav nlp\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'sentencepiece']' returned non-zero exit status 1.
Error output: d:\padhai\gaurav nlp\.venv\Scripts\python.exe: No module named pip


 Installation failed. Please try manually:
pip install sentencepiece

Restarting kernel is recommended after installing new packages...


In [1]:
# Core imports
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')


from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from datasets import Dataset
import transformers

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Libraries imported successfully!
PyTorch version: 2.8.0+cu129
Transformers version: 4.57.0
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.0 GB


In [2]:
# Configuration for T5 (OPTIMIZED FOR FASTER TRAINING)
CONFIG = {
    # Model settings - Using multilingual T5
    'model_name': 'google/mt5-small',  # Changed to valid model
    'max_length': 128,  # REDUCED from 256 for faster training
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # Data settings
    'dataset_path': './Hindi/train.csv',
    'test_size': 0.2,  # 20% for dev set
    'random_seed': 42,
    
    # Training settings - OPTIMIZED FOR SPEED
    'output_dir': './mt5-hindi-final-v4',
    'num_epochs': 3,  # REDUCED from 50 for faster training
    'train_batch_size': 8,  # INCREASED from 4 for faster training
    'eval_batch_size': 8,   # Match train batch size
    'gradient_accumulation_steps': 4,  # REDUCED from 16 but maintains reasonable effective batch size
    'learning_rate': 1e-4,  # INCREASED from 5e-5 for faster convergence
    'warmup_ratio': 0.05,  # REDUCED from 0.1
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    'lr_scheduler_type': 'linear',
    
    # Init/Resume settings
    'init_model_dir': './mt5-hindi-final-v3',
    'resume_from_checkpoint': True,  # Re-enabled for faster training
    'resume_checkpoint_path': './mt5-hindi-final-v3/checkpoint-1000',
    'ignore_missing_keys': True,
    'strict_loading': False,
    
    # Augmentation / noise injection settings - REDUCED FOR SPEED
    'augment': {
        'enabled': True,
        'ratio': 0.2,  # REDUCED from 0.5 to create less synthetic data
        'weights': {
            'postposition': 0.45,
            'matra': 0.35,
            'punctuation': 0.12,
            'typo': 0.08
        },
        'per_char_matra_prob': 0.12,
        'max_tries_per_sample': 2,  # REDUCED from 3
        'identity_boost_ratio': 0.1  # REDUCED from 0.2
    },
    
    # Evaluation settings - OPTIMIZED FOR SPEED
    'eval_steps': 100,  # REDUCED from 250 for more frequent but faster evals
    'save_steps': 100,  # REDUCED from 250 for more frequent saves
    'logging_steps': 25,  # REDUCED from 50
    'save_total_limit': 2,  # REDUCED from 3 to save disk space
    'early_stopping_patience': 5,  # INCREASED from 3 to allow more training
    
    # Generation settings for T5
    'generation_config': {
        'max_new_tokens': 64,  # REDUCED from 128
        'min_new_tokens': 1,
        'num_beams': 2,  # REDUCED from 4 for faster inference
        'early_stopping': True,
        'repetition_penalty': 1.2,
        'length_penalty': 1.0,
        'do_sample': False,
    }
}

print(" OPTIMIZED Configuration for FASTER Training:")
for key, value in CONFIG.items():
    if key not in ('generation_config', 'augment'):
        print(f"   {key}: {value}")

print(f"\n Generation Config (OPTIMIZED):")
for key, value in CONFIG['generation_config'].items():
    print(f"   {key}: {value}")

print(f"\n Augmentation Config (REDUCED):")
for key, value in CONFIG['augment'].items():
    print(f"   {key}: {value}")


effective_batch_size = CONFIG['train_batch_size'] * CONFIG['gradient_accumulation_steps']
print(f"\n Performance Estimates:")
print(f"   Effective batch size: {effective_batch_size}")
print(f"   Expected speedup: ~8-10x faster than original config")
print(f"   Estimated training time: ~3-4 hours (vs 30+ hours original)")



 OPTIMIZED Configuration for FASTER Training:
   model_name: google/mt5-small
   max_length: 128
   device: cuda
   dataset_path: ./Hindi/train.csv
   test_size: 0.2
   random_seed: 42
   output_dir: ./mt5-hindi-final-v4
   num_epochs: 3
   train_batch_size: 8
   eval_batch_size: 8
   gradient_accumulation_steps: 4
   learning_rate: 0.0001
   warmup_ratio: 0.05
   weight_decay: 0.01
   max_grad_norm: 1.0
   lr_scheduler_type: linear
   init_model_dir: ./mt5-hindi-final-v3
   resume_from_checkpoint: True
   resume_checkpoint_path: ./mt5-hindi-final-v3/checkpoint-1000
   ignore_missing_keys: True
   strict_loading: False
   eval_steps: 100
   save_steps: 100
   logging_steps: 25
   save_total_limit: 2
   early_stopping_patience: 5

 Generation Config (OPTIMIZED):
   max_new_tokens: 64
   min_new_tokens: 1
   num_beams: 2
   early_stopping: True
   repetition_penalty: 1.2
   length_penalty: 1.0
   do_sample: False

 Augmentation Config (REDUCED):
   enabled: True
   ratio: 0.2
   weights:

In [3]:
# Load and prepare dataset
print("Loading dataset...")

# Ensure CONFIG is defined before using it
if 'CONFIG' not in globals():
    raise NameError("The 'CONFIG' dictionary is not defined. Please run the configuration cell first.")

dataset_path = Path(CONFIG['dataset_path'])
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset not found: {dataset_path}")

# Load dataset
df = pd.read_csv(dataset_path, encoding='utf-8')
print(f"Dataset loaded: {df.shape}")
print(f"File size: {dataset_path.stat().st_size / (1024**2):.1f} MB")
print(f"Columns: {list(df.columns)}")

# Handle column names
if 'input' in df.columns and 'output' in df.columns:
    input_col, output_col = 'input', 'output'
elif len(df.columns) >= 2:
    input_col, output_col = df.columns[0], df.columns[1]
    print(f"Using columns: '{input_col}' -> '{output_col}'")
else:
    raise ValueError("Cannot identify input/output columns")

# Clean data
print(f"\nCleaning data...")
original_size = len(df)

# Remove nulls and empty strings
df = df.dropna(subset=[input_col, output_col])
df[input_col] = df[input_col].astype(str).str.strip()
df[output_col] = df[output_col].astype(str).str.strip()
df = df[(df[input_col] != '') & (df[output_col] != '')]

# Filter by length (5-150 characters for T5)
df = df[
    (df[input_col].str.len() >= 5) & (df[input_col].str.len() <= 400) &
    (df[output_col].str.len() >= 5) & (df[output_col].str.len() <= 400)
]

print(f"   Original: {original_size:,} samples")
print(f"   Cleaned: {len(df):,} samples")
print(f"   Removed: {original_size - len(df):,} samples")

# Rename columns
df_clean = df[[input_col, output_col]].copy()
df_clean.columns = ['input_text', 'output_text']

# Analyze data composition
identical = (df_clean['input_text'] == df_clean['output_text']).sum()
corrections = len(df_clean) - identical

print(f"\nData composition:")
print(f"   Identity pairs: {identical:,} ({identical/len(df_clean)*100:.1f}%)")
print(f"   Corrections: {corrections:,} ({corrections/len(df_clean)*100:.1f}%)")

# Show examples
print(f"\nSample corrections:")
correction_samples = df_clean[df_clean['input_text'] != df_clean['output_text']].head(3)
for i, (_, row) in enumerate(correction_samples.iterrows(), 1):
    print(f"  {i}. Input:  {row['input_text'][:80]}...")
    print(f"     Output: {row['output_text'][:80]}...")
    print()

Loading dataset...
Dataset loaded: (13759, 3)
File size: 5.2 MB
Columns: ['Input sentence', 'Output sentence', 'Unnamed: 2']
Using columns: 'Input sentence' -> 'Output sentence'

Cleaning data...
   Original: 13,759 samples
   Cleaned: 13,758 samples
   Removed: 1 samples

Data composition:
   Identity pairs: 58 (0.4%)
   Corrections: 13,700 (99.6%)

Sample corrections:
  1. Input:  चाय की दुकान से लेकर वाहनों और दिवारों तक हर जगह विज्ञापन ही विज्ञापन दिखाई देते...
     Output: चाय की दुकान से लेकर वाहनों और दिवारों तक हर जगह विज्ञापन ही विज्ञापन दिखाई देते...

  2. Input:  ये कहीं पे निगाहें , कही पे निशाना का सा अन्दाज है ।...
     Output: यह कहीं पे निगाहें , कही पे निशाना का सा अन्दाज है ।...

  3. Input:  आज हम विज्ञापन युग के सीमान्त पर आ खड़े हुए है ।...
     Output: आज हम विज्ञापन युग के सीमान्त पर आ खड़े हुए हैं ।...



In [4]:
# Augmentation: Hindi noise injection utilities
import random
random.seed(CONFIG['random_seed'])
np.random.seed(CONFIG['random_seed'])

# Mapping for common Hindi postpositions and variants to perturb
POSTPOSITIONS = ["में", "पर", "से", "को", "का", "की", "के", "तक", "लिए", "साथ", "बारे", "द्वारा", "ऊपर", "नीचे", "वाले"]

# Simple punctuation set and noise options
PUNCT = [".", ",", "?", "!", "…", "—", "-", ":", ";", "।"]

# Basic keyboard-adjacent typos for Devanagari (approximate pairs)
TYPO_MAP = {
    "क": ["ख", "ग"], "ख": ["क", "घ"],
    "ग": ["घ", "क"], "घ": ["ग", "ङ"],
    "ज": ["झ", "ज"], "ट": ["ठ", "ड"],
    "त": ["थ", "द"], "द": ["ध", "त"],
    "न": ["ण", "म"], "प": ["फ", "ब"],
    "ब": ["भ", "व"], "य": ["र", "ह"],
    "ा": ["ि", "ी"], "ि": ["ी", "ु"],
    "ी": ["ि", "े"], "ु": ["ू", "ो"],
    "ू": ["ु", "ौ"], "े": ["ै", "ि"],
    "ै": ["े", "ौ"], "ो": ["ौ", "ु"],
    "ौ": ["ो", "ू"],
}

# Matra characters and a swap ring to simulate errors
MATRA_RING = ["ा", "ि", "ी", "ु", "ू", "े", "ै", "ो", "ौ"]
MATRA_IDX = {m: i for i, m in enumerate(MATRA_RING)}


def swap_matra(text: str, per_char_prob: float = 0.08) -> str:
    chars = list(text)
    for i, ch in enumerate(chars):
        if ch in MATRA_IDX and random.random() < per_char_prob:
            idx = MATRA_IDX[ch]
            # move +/-1 in ring
            if random.random() < 0.5:
                new_ch = MATRA_RING[(idx + 1) % len(MATRA_RING)]
            else:
                new_ch = MATRA_RING[(idx - 1) % len(MATRA_RING)]
            chars[i] = new_ch
    return "".join(chars)


def inject_typo(text: str, prob: float = 0.03) -> str:
    chars = list(text)
    for i, ch in enumerate(chars):
        if ch in TYPO_MAP and random.random() < prob:
            repl = random.choice(TYPO_MAP[ch])
            chars[i] = repl
    return "".join(chars)


def perturb_postposition(text: str, change_prob: float = 0.6) -> str:
    words = text.split()
    for i, w in enumerate(words):
        if w in POSTPOSITIONS and random.random() < change_prob:
            # swap to a nearby/related postposition
            candidates = [p for p in POSTPOSITIONS if p != w]
            if candidates:
                words[i] = random.choice(candidates)
    return " ".join(words)


def perturb_punctuation(text: str, add_prob: float = 0.25, drop_prob: float = 0.15, swap_prob: float = 0.1) -> str:
    # Drop some existing punctuation
    out = []
    for ch in text:
        if ch in PUNCT and random.random() < drop_prob:
            continue
        out.append(ch)
    text2 = "".join(out)
    # Randomly add punctuation near clause boundaries (spaces)
    tokens = text2.split(" ")
    for i in range(len(tokens)-1):
        if random.random() < add_prob:
            tokens[i] = tokens[i] + random.choice(PUNCT)
    text3 = " ".join(tokens)
    # Occasionally swap one punctuation to another
    out = []
    for ch in text3:
        if ch in PUNCT and random.random() < swap_prob:
            out.append(random.choice(PUNCT))
        else:
            out.append(ch)
    return "".join(out)


def apply_noise(text: str, weights: dict, per_char_matra_prob: float) -> str:
    # Weighted choice of which primary error to apply, optional stacking
    r = random.random()
    cum = 0.0
    choice = 'postposition'
    for k in ['postposition','matra','punctuation','typo']:
        cum += weights.get(k, 0)
        if r <= cum:
            choice = k
            break
    noisy = text
    if choice == 'postposition':
        noisy = perturb_postposition(noisy)
        # stack small matra or punctuation noise
        noisy = swap_matra(noisy, per_char_prob=per_char_matra_prob*0.5)
        noisy = perturb_punctuation(noisy, add_prob=0.15, drop_prob=0.05, swap_prob=0.05)
    elif choice == 'matra':
        noisy = swap_matra(noisy, per_char_prob=per_char_matra_prob)
        noisy = inject_typo(noisy, prob=0.01)
    elif choice == 'punctuation':
        noisy = perturb_punctuation(noisy)
    elif choice == 'typo':
        noisy = inject_typo(noisy, prob=0.05)
        noisy = swap_matra(noisy, per_char_prob=per_char_matra_prob*0.5)
    return noisy


def make_synthetic_pairs(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    aug_cfg = config.get('augment', {})
    if not aug_cfg.get('enabled', False):
        return pd.DataFrame(columns=['input_text','output_text'])
    ratio = aug_cfg.get('ratio', 0.5)
    weights = aug_cfg.get('weights', {})
    per_char_matra_prob = aug_cfg.get('per_char_matra_prob', 0.08)
    max_tries = aug_cfg.get('max_tries_per_sample', 3)
    n_target = int(len(df) * ratio)
    # Prefer sampling from correct outputs to inject errors into clean text
    pool = df.copy().reset_index(drop=True)
    samples = []
    tries = 0
    i = 0
    while i < n_target and tries < n_target * 5:
        row = pool.iloc[random.randrange(len(pool))]
        clean = row['output_text']
        noisy = clean
        attempt = 0
        while attempt < max_tries and noisy.strip() == clean.strip():
            noisy = apply_noise(clean, weights, per_char_matra_prob)
            attempt += 1
        if noisy.strip() != clean.strip():
            samples.append({'input_text': noisy, 'output_text': clean})
            i += 1
        tries += 1
    return pd.DataFrame(samples)

print("Augmentation utilities ready (postposition, matra, punctuation, typo)")

Augmentation utilities ready (postposition, matra, punctuation, typo)


In [5]:
# Create train/dev splits
print("Creating train/dev splits...")

# Stratified split to maintain correction ratio
df_clean['is_correction'] = (df_clean['input_text'] != df_clean['output_text']).astype(int)

train_df, dev_df = train_test_split(
    df_clean,
    test_size=CONFIG['test_size'],
    random_state=CONFIG['random_seed'],
    stratify=df_clean['is_correction']
)

# Remove helper column
train_df = train_df[['input_text', 'output_text']].reset_index(drop=True)
dev_df = dev_df[['input_text', 'output_text']].reset_index(drop=True)

print(f"Split complete:")
print(f"   Train: {len(train_df):,} samples")
print(f"   Dev: {len(dev_df):,} samples")

# Verify split distribution
train_corrections = (train_df['input_text'] != train_df['output_text']).sum()
dev_corrections = (dev_df['input_text'] != dev_df['output_text']).sum()

print(f"\nDistribution maintained:")
print(f"   Train corrections: {train_corrections:,}/{len(train_df):,} ({train_corrections/len(train_df)*100:.1f}%)")
print(f"   Dev corrections: {dev_corrections:,}/{len(dev_df):,} ({dev_corrections/len(dev_df)*100:.1f}%)")

# ===== Apply augmentation to training split =====
if CONFIG.get('augment', {}).get('enabled', False):
    print("\nApplying synthetic noise augmentation to match dev distribution...")
    synth_df = make_synthetic_pairs(train_df, CONFIG)
    print(f"   Synthesized: {len(synth_df):,} samples")

    # Optional: identity boost to teach the model to leave correct sentences untouched
    id_boost = CONFIG['augment'].get('identity_boost_ratio', 0.0)
    if id_boost > 0:
        identity_df = train_df[train_df['input_text'] == train_df['output_text']]
        n_boost = max(1, int(len(train_df) * id_boost))
        if len(identity_df) > 0:
            identity_boost_df = identity_df.sample(
                n=min(n_boost, len(identity_df)),
                replace=(n_boost > len(identity_df)),
                random_state=CONFIG['random_seed']
            ).copy()
            print(f"   Identity boost: +{len(identity_boost_df)} copies of correct pairs")
        else:
            identity_boost_df = pd.DataFrame(columns=['input_text', 'output_text'])
            print("   Identity boost skipped (no identity pairs found)")
    else:
        identity_boost_df = pd.DataFrame(columns=['input_text', 'output_text'])

    # Combine original + synthetic + identity boost
    train_df_aug = pd.concat([train_df, synth_df, identity_boost_df], ignore_index=True)
    # Shuffle for mixing
    train_df_aug = train_df_aug.sample(frac=1.0, random_state=CONFIG['random_seed']).reset_index(drop=True)
    print(f"   Augmented train size: {len(train_df_aug):,}")
    # Show composition
    ident_aug = (train_df_aug['input_text'] == train_df_aug['output_text']).sum()
    corr_aug = len(train_df_aug) - ident_aug
    print(f"   Augmented composition -> Identity: {ident_aug:,} ({ident_aug/len(train_df_aug)*100:.1f}%), Corrections: {corr_aug:,} ({corr_aug/len(train_df_aug)*100:.1f}%)")
else:
    train_df_aug = train_df.copy()

# Keep a snapshot (optional)
train_snapshot_path = Path(CONFIG['output_dir']) / 'train_aug_snapshot.csv'
train_snapshot_path.parent.mkdir(parents=True, exist_ok=True)
train_df_aug.to_csv(train_snapshot_path, index=False, encoding='utf-8')
print(f"   Saved augmented train snapshot -> {train_snapshot_path}")

Creating train/dev splits...
Split complete:
   Train: 11,006 samples
   Dev: 2,752 samples

Distribution maintained:
   Train corrections: 10,960/11,006 (99.6%)
   Dev corrections: 2,740/2,752 (99.6%)

Applying synthetic noise augmentation to match dev distribution...
   Synthesized: 2,201 samples
   Identity boost: +46 copies of correct pairs
   Augmented train size: 13,253
   Augmented composition -> Identity: 92 (0.7%), Corrections: 13,161 (99.3%)
   Saved augmented train snapshot -> mt5-hindi-final-v4\train_aug_snapshot.csv


In [6]:
# Load mT5 model and tokenizer (multilingual T5)
print(f"Loading mT5 model and tokenizer...")

# Clear GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Determine init source based on configuration
if CONFIG.get('resume_from_checkpoint', False):
    init_source = CONFIG.get('init_model_dir') if Path(CONFIG.get('init_model_dir','')).exists() else CONFIG['model_name']
else:
    init_source = CONFIG['model_name']  # Start fresh from base model
    print(" Starting fresh training - using base model instead of checkpoint")

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from pathlib import Path

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "google/mt5-small", 
    use_fast=False,
    legacy=False,  # Recommended for new T5/mT5 training
    local_files_only=False  # Ensure it can download if needed
)
print(f"Tokenizer loaded: {type(tokenizer).__name__}")
print(f"   Vocab size: {len(tokenizer):,}")
print(f"   Init source: {init_source}")

# Load model with comprehensive error handling
def load_model_safely(source_path, base_model_name):
    """Load model with fallback and missing key handling"""
    try:
        print(f" Attempting to load from: {source_path}")
        model = AutoModelForSeq2SeqLM.from_pretrained(
            source_path,
            torch_dtype=torch.float32,  # Use float32 for stability
            ignore_mismatched_sizes=True,  # Handle size mismatches
            local_files_only=False
        )
        print(f" Model loaded successfully from checkpoint")
        return model, True
    except Exception as e:
        print(f"  Warning loading from checkpoint: {str(e)}")
        if "missing keys" in str(e).lower() or "embed_tokens" in str(e):
            print(" Detected missing embedding keys - this is a known issue")
        
        print(f" Falling back to base model: {base_model_name}")
        try:
            model = AutoModelForSeq2SeqLM.from_pretrained(
                base_model_name,
                torch_dtype=torch.float32
            )
            print(f" Base model loaded successfully")
            return model, False
        except Exception as base_e:
            print(f" Failed to load base model: {str(base_e)}")
            raise base_e

model, loaded_from_checkpoint = load_model_safely(init_source, CONFIG['model_name'])

# Handle embedding tokens and ensure proper setup
print(f"\n Model setup and validation...")

# Check and handle embedding dimensions
if hasattr(model.config, 'vocab_size') and len(tokenizer) != model.config.vocab_size:
    print(f" Vocab size mismatch detected:")
    print(f"   Tokenizer vocab: {len(tokenizer):,}")
    print(f"   Model vocab: {model.config.vocab_size:,}")
    print(f" Resizing model embeddings to match tokenizer...")
    model.resize_token_embeddings(len(tokenizer))
    print(f" Embeddings resized successfully")

# Verify embedding layers exist and are trainable
embedding_status = []
if hasattr(model, 'encoder') and hasattr(model.encoder, 'embed_tokens'):
    trainable = model.encoder.embed_tokens.weight.requires_grad
    embedding_status.append(f"Encoder embeddings: {' trainable' if trainable else '  frozen'}")
else:
    embedding_status.append("Encoder embeddings:  not found")

if hasattr(model, 'decoder') and hasattr(model.decoder, 'embed_tokens'):
    trainable = model.decoder.embed_tokens.weight.requires_grad
    embedding_status.append(f"Decoder embeddings: {' trainable' if trainable else '  frozen'}")
else:
    embedding_status.append("Decoder embeddings:  not found")

for status in embedding_status:
    print(f"   {status}")

# Tie embeddings to ensure consistency
if hasattr(model, 'tie_weights'):
    try:
        model.tie_weights()
        print(f" Embeddings tied successfully")
    except Exception as tie_e:
        print(f"  Embedding tie warning: {str(tie_e)}")

# Fix tokenizer padding configuration
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    if hasattr(model, 'config'):
        model.config.pad_token_id = tokenizer.eos_token_id
    print(f" Pad token configured")

print(f"\n Model summary:")
print(f"   Model type: {type(model).__name__}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"   Device target: {CONFIG['device']}")
print(f"   Loaded from checkpoint: {' Yes' if loaded_from_checkpoint else ' No (base model)'}")

# Move model to device
model = model.to(CONFIG['device'])
print(f" Model moved to {CONFIG['device']}")

if CONFIG['device'] == 'cuda':
    print(f"  GPU memory: {torch.cuda.memory_allocated() / 1024**3:.1f} GB allocated")
    print(f"  GPU memory: {torch.cuda.memory_reserved() / 1024**3:.1f} GB reserved")

print(f"\n Model ready for training!")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /google/mt5-small/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002426E962510>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: a29fd73d-d6c5-486d-9844-a4624a78c72b)')' thrown while requesting HEAD https://huggingface.co/google/mt5-small/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


Loading mT5 model and tokenizer...


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /google/mt5-small/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002426E99DF90>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 8e7bf102-b65c-4628-913f-855fd253f393)')' thrown while requesting HEAD https://huggingface.co/google/mt5-small/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /google/mt5-small/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002426E99E5D0>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 1fdd7852-3971-438a-af22-a4df32e44e1b)')' thrown while requesting HEAD https://huggingface.co/google/mt5-small/resolve/main/tokenize

Tokenizer loaded: T5Tokenizer
   Vocab size: 250,100
   Init source: google/mt5-small
 Attempting to load from: google/mt5-small


`torch_dtype` is deprecated! Use `dtype` instead!


 Model loaded successfully from checkpoint

 Model setup and validation...
 Vocab size mismatch detected:
   Tokenizer vocab: 250,100
   Model vocab: 250,112
 Resizing model embeddings to match tokenizer...
 Embeddings resized successfully
   Encoder embeddings:  trainable
   Decoder embeddings:  trainable
 Embeddings tied successfully

 Model summary:
   Model type: MT5ForConditionalGeneration
   Parameters: 300.2M
   Device target: cuda
   Loaded from checkpoint:  Yes
 Model moved to cuda
  GPU memory: 1.1 GB allocated
  GPU memory: 1.1 GB reserved

 Model ready for training!


In [7]:
# Tokenization function for T5 (FIXED for OverflowError)
def tokenize_function(examples):
    """Tokenize examples for T5 - uses text-to-text format with proper label handling"""
    # Create T5-style input with task prefix
    inputs = [f"grammar: {text}" for text in examples['input_text']]
    targets = examples['output_text']
    
    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=CONFIG['max_length'],
        truncation=True,
        padding=False,  # Dynamic padding during training
        return_tensors=None
    )
    
    # Tokenize targets - T5 handles this differently
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=CONFIG['max_length'],
            truncation=True,
            padding=False,
            return_tensors=None
        )
    
    # CRITICAL FIX: Handle labels properly to avoid OverflowError
    model_inputs["labels"] = labels["input_ids"]
    
    # Ensure all values are valid integers (no negative values that could cause overflow)
    for i, label_seq in enumerate(model_inputs["labels"]):
        # Convert to list if needed and ensure all values are valid
        if isinstance(label_seq, list):
            model_inputs["labels"][i] = [max(0, int(token_id)) for token_id in label_seq]
        else:
            model_inputs["labels"][i] = [max(0, int(token_id)) for token_id in label_seq.tolist()]
    
    return model_inputs

print("✅ T5 tokenization function created (FIXED for OverflowError)")

# Test tokenization with error checking
print("\nTesting tokenization with error prevention...")
if 'train_df_aug' in globals() and len(train_df_aug) > 0:
    test_example = {
        'input_text': [train_df_aug.iloc[0]['input_text']],
        'output_text': [train_df_aug.iloc[0]['output_text']]
    }
    
    try:
        test_result = tokenize_function(test_example)
        print(f"✅ Tokenization test passed!")
        print(f"   Keys: {list(test_result.keys())}")
        print(f"   Input IDs length: {len(test_result['input_ids'][0])}")
        print(f"   Labels length: {len(test_result['labels'][0])}")
        
        # Check for negative values that could cause OverflowError
        input_min = min(test_result['input_ids'][0])
        label_min = min(test_result['labels'][0])
        
        print(f"   Min input ID: {input_min} (should be >= 0)")
        print(f"   Min label ID: {label_min} (should be >= 0)")
        
        if input_min >= 0 and label_min >= 0:
            print(f"✅ No negative token IDs - OverflowError should be fixed!")
        else:
            print(f"⚠️  Warning: Found negative token IDs")
            
    except Exception as e:
        print(f"❌ Tokenization test failed: {str(e)}")
else:
    print("⚠️  Cannot test - train_df_aug not available")

# Additional safety check function
def validate_tokenized_dataset(dataset, name="dataset"):
    """Validate tokenized dataset for potential OverflowError issues"""
    print(f"\n🔍 Validating {name} for OverflowError issues...")
    
    try:
        sample = dataset[0]
        
        # Check input_ids
        input_ids = sample['input_ids']
        if any(id < 0 for id in input_ids):
            print(f"❌ Found negative input_ids in {name}")
            return False
        
        # Check labels
        labels = sample['labels']
        if any(id < 0 for id in labels):
            print(f"❌ Found negative labels in {name}")
            return False
            
        print(f"✅ {name} validation passed - no negative token IDs")
        return True
        
    except Exception as e:
        print(f"❌ Error validating {name}: {str(e)}")
        return False

print("\n✅ Tokenization function fixed for OverflowError!")
print("Re-run the dataset creation cells to apply the fix.")

✅ T5 tokenization function created (FIXED for OverflowError)

Testing tokenization with error prevention...
✅ Tokenization test passed!
   Keys: ['input_ids', 'attention_mask', 'labels']
   Input IDs length: 41
   Labels length: 38
   Min input ID: 1 (should be >= 0)
   Min label ID: 1 (should be >= 0)
✅ No negative token IDs - OverflowError should be fixed!

✅ Tokenization function fixed for OverflowError!
Re-run the dataset creation cells to apply the fix.


In [8]:
# Create datasets
print("Creating datasets...")

# Convert to HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df_aug)  # use augmented
dev_dataset = Dataset.from_pandas(dev_df)

print(f"   Created train dataset: {len(train_dataset):,} samples")
print(f"   Created dev dataset: {len(dev_dataset):,} samples")

# Tokenize datasets
print(f"\nTokenizing datasets...")

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train"
)

dev_tokenized = dev_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dev_dataset.column_names,
    desc="Tokenizing dev"
)

print(f"Tokenization complete!")
print(f"   Train tokenized: {len(train_tokenized):,} samples")
print(f"   Dev tokenized: {len(dev_tokenized):,} samples")

# Show tokenized example
print(f"\nTokenized sample:")
sample = train_tokenized[0]
print(f"   Keys: {list(sample.keys())}")
print(f"   Input IDs: {sample['input_ids'][:10]}... (length: {len(sample['input_ids'])})")
print(f"   Labels: {sample['labels'][:10]}... (length: {len(sample['labels'])})")

# Verify tokenized datasets are T5 compatible
if 'token_type_ids' not in sample:
    print(f"✓ Tokenized datasets are T5 compatible!")
else:
    print(f"Warning: token_type_ids found in dataset - unexpected for T5")

Creating datasets...
   Created train dataset: 13,253 samples
   Created dev dataset: 2,752 samples

Tokenizing datasets...


Tokenizing dev: 100%|██████████| 2752/2752 [00:00<00:00, 5455.42 examples/s]

Tokenization complete!
   Train tokenized: 13,253 samples
   Dev tokenized: 2,752 samples

Tokenized sample:
   Keys: ['input_ids', 'attention_mask', 'labels']
   Input IDs: [259, 164814, 267, 1822, 3597, 29489, 50681, 274, 259, 28031]... (length: 41)
   Labels: [1822, 3597, 29489, 50681, 274, 259, 28031, 3689, 14263, 36158]... (length: 38)
✓ Tokenized datasets are T5 compatible!


In [9]:
# Create data collator (FIXED for OverflowError)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt",
    label_pad_token_id=-100  # CRITICAL: Explicitly set label padding token
)

print(f"   Data collator created: {type(data_collator).__name__}")
print(f"   Label pad token ID: {data_collator.label_pad_token_id}")

# Test data collator with safety checks
print(f"\n Testing data collator with OverflowError prevention...")
try:
    test_batch = [train_tokenized[i] for i in range(2)]  # Get 2 samples
    
    # Pre-check the test batch for negative values
    print("Pre-flight check of test batch...")
    for i, sample in enumerate(test_batch):
        input_min = min(sample['input_ids'])
        label_min = min(sample['labels'])
        print(f"   Sample {i}: input_min={input_min}, label_min={label_min}")
        
        if input_min < 0 or label_min < 0:
            print(f"  Warning: Sample {i} has negative token IDs")
    
    # Test collation
    collated_batch = data_collator(test_batch)
    
    print(f" Data collator test passed!")
    print(f"   Batch keys: {list(collated_batch.keys())}")
    print(f"   Input IDs shape: {collated_batch['input_ids'].shape}")
    print(f"   Labels shape: {collated_batch['labels'].shape}")
    
    # Check for negative values in collated batch
    input_min_val = collated_batch['input_ids'].min().item()
    label_min_val = collated_batch['labels'].min().item()
    
    print(f"   Collated input min: {input_min_val}")
    print(f"   Collated label min: {label_min_val}")
    
    if input_min_val >= -100 and label_min_val >= -100:  # -100 is acceptable for labels
        print(f" Collated batch looks good - OverflowError should be prevented!")
    else:
        print(f"  Warning: Unusual values in collated batch")
    
    # Verify collated batch is T5 compatible
    if 'token_type_ids' not in collated_batch:
        print(f" Collated batch is T5 compatible!")
    else:
        print(f"  Warning: token_type_ids in collated batch - unexpected for T5")
        
except Exception as e:
    print(f" Data collator test failed: {str(e)}")
    print(f"This might be the source of your OverflowError!")
    
    # Provide debugging info
    print(f"\n Debugging information:")
    if 'train_tokenized' in globals():
        sample = train_tokenized[0]
        print(f"   Sample keys: {list(sample.keys())}")
        print(f"   Input IDs type: {type(sample['input_ids'])}")
        print(f"   Labels type: {type(sample['labels'])}")
        print(f"   Input IDs sample: {sample['input_ids'][:5]}")
        print(f"   Labels sample: {sample['labels'][:5]}")

print(f"\n Data collator configured with OverflowError prevention!")

   Data collator created: DataCollatorForSeq2Seq
   Label pad token ID: -100

 Testing data collator with OverflowError prevention...
Pre-flight check of test batch...
   Sample 0: input_min=1, label_min=1
   Sample 1: input_min=1, label_min=1
 Data collator test passed!
   Batch keys: ['input_ids', 'attention_mask', 'labels', 'decoder_input_ids']
   Input IDs shape: torch.Size([2, 41])
   Labels shape: torch.Size([2, 38])
   Collated input min: 0
   Collated label min: -100
 Collated batch looks good - OverflowError should be prevented!
 Collated batch is T5 compatible!

 Data collator configured with OverflowError prevention!


In [10]:
from transformers import Seq2SeqTrainingArguments, GenerationConfig, set_seed

set_seed(CONFIG['random_seed'])

training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['train_batch_size'],
    per_device_eval_batch_size=CONFIG['eval_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    warmup_ratio=CONFIG['warmup_ratio'],
    max_grad_norm=CONFIG['max_grad_norm'],
    lr_scheduler_type=CONFIG['lr_scheduler_type'],

    eval_strategy="steps",
    eval_steps=CONFIG['eval_steps'],
    save_strategy="steps",
    save_steps=CONFIG['save_steps'],
    save_total_limit=CONFIG['save_total_limit'],
    logging_steps=CONFIG['logging_steps'],

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",   # or your custom metric key
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=CONFIG['max_length'],
    report_to=["none"],                  # optional, to silence HF loggers
)

gen_config = GenerationConfig(**CONFIG['generation_config'])

# Later:
# trainer = Seq2SeqTrainer(..., args=training_args, ...)
# trainer.train(resume_from_checkpoint=CONFIG['resume_checkpoint_path'] if CONFIG['resume_from_checkpoint'] else None)

In [12]:
from transformers import Seq2SeqTrainingArguments, GenerationConfig, set_seed

set_seed(CONFIG['random_seed'])

training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['train_batch_size'],
    per_device_eval_batch_size=CONFIG['eval_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    warmup_ratio=CONFIG['warmup_ratio'],
    max_grad_norm=CONFIG['max_grad_norm'],
    lr_scheduler_type=CONFIG['lr_scheduler_type'],

    eval_strategy="steps",  # Fixed: renamed from evaluation_strategy in newer transformers
    eval_steps=CONFIG['eval_steps'],
    save_strategy="steps",
    save_steps=CONFIG['save_steps'],
    save_total_limit=CONFIG['save_total_limit'],
    logging_steps=CONFIG['logging_steps'],

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",   # or your custom metric key
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=CONFIG['max_length'],
    report_to=["none"],                  # optional, to silence HF loggers
)

gen_config = GenerationConfig(**CONFIG['generation_config'])

# Later:
# trainer = Seq2SeqTrainer(..., args=training_args, ...)
# trainer.train(resume_from_checkpoint=CONFIG['resume_checkpoint_path'] if CONFIG['resume_from_checkpoint'] else None)

# The following block should be inside a function, for example:
def check_token_ids():
    if 'train_tokenized' in globals():
        print("Checking train dataset...")
        sample = train_tokenized[0]
        input_ids = sample['input_ids']
        labels = sample['labels']

        if any(id < 0 for id in input_ids):
            print(" Found negative input_ids in train dataset!")
            return False

        if any(id < 0 for id in labels):
            print(" Found negative labels in train dataset!")
            return False

        print(" Train dataset token IDs look good")

    if 'dev_tokenized' in globals():
        print("Checking dev dataset...")
        sample = dev_tokenized[0]
        input_ids = sample['input_ids']
        labels = sample['labels']

        if any(id < 0 for id in input_ids):
            print(" Found negative input_ids in dev dataset!")
            return False

        if any(id < 0 for id in labels):
            print(" Found negative labels in dev dataset!")
            return False

        print(" Dev dataset token IDs look good")

    # Fix 4: Test data collator with safety
    print(" Testing data collator safety...")
    if 'data_collator' in globals() and 'train_tokenized' in globals():
        try:
            test_sample = [train_tokenized[0]]
            test_batch = data_collator(test_sample)

            # Check the collated batch
            input_min = test_batch['input_ids'].min().item()
            label_min = test_batch['labels'].min().item()

            print(f"   Collated input min: {input_min}")
            print(f"   Collated label min: {label_min}")

            if input_min < -100 or label_min < -200:  # Allow some negative values for special tokens
                print(" Suspicious values in collated batch!")
                return False
            else:
                print(" Data collator produces safe values")

        except Exception as e:
            print(f" Data collator test failed: {str(e)}")
            return False

    # Fix 5: Check training arguments for potential issues
    print(" Checking training arguments...")
    if 'training_args' in globals():
        # Ensure no problematic settings
        if hasattr(training_args, 'fp16') and training_args.fp16:
            print("   FP16 enabled - this can sometimes cause overflow issues")
            print("   Consider disabling fp16 if problems persist")

        if hasattr(training_args, 'dataloader_num_workers'):
            if training_args.dataloader_num_workers > 0:
                print("  Multiple dataloader workers can cause issues")
                print("   Consider setting dataloader_num_workers=0")

    print(f"\n OverflowError prevention complete!")
    # fixes_applied should be defined somewhere in your code
    if 'fixes_applied' in globals() and fixes_applied:
        print(f"Fixes applied: {', '.join(fixes_applied)}")
    else:
        print("No fixes needed - configuration looks good")

    return True

# Call the function
check_token_ids()

def debug_overflow_error(error_msg=""):
    """Debug OverflowError with detailed information"""
    print(f"\n OverflowError Debugging")
    print(f"Error message: {error_msg}")
    print("-" * 40)
    
    # Check tokenizer state
    print(f"Tokenizer info:")
    print(f"   Vocab size: {len(tokenizer)}")
    print(f"   Pad token ID: {tokenizer.pad_token_id}")
    print(f"   EOS token ID: {tokenizer.eos_token_id}")
    print(f"   UNK token ID: {tokenizer.unk_token_id}")
    
    # Check model state
    if 'model' in globals():
        print(f"\nModel info:")
        print(f"   Config vocab size: {getattr(model.config, 'vocab_size', 'Unknown')}")
        print(f"   Config pad token ID: {getattr(model.config, 'pad_token_id', 'Unknown')}")
        print(f"   Model dtype: {model.dtype}")
    
    # Check datasets
    if 'train_tokenized' in globals():
        sample = train_tokenized[0]
        print(f"\nTrain dataset sample:")
        print(f"   Input IDs range: {min(sample['input_ids'])} to {max(sample['input_ids'])}")
        print(f"   Labels range: {min(sample['labels'])} to {max(sample['labels'])}")

# Run the fix
# success = fix_overflow_error()

# if not success:
#     print(f"\n OverflowError prevention failed!")
#     print(f"You may need to re-create your datasets with the fixed tokenization function.")
#     print(f"Steps to fix:")
#     print(f"   1. Re-run the tokenization function cell")
#     print(f"   2. Re-run the dataset creation cell")
#     print(f"   3. Re-run the data collator cell")
#     print(f"   4. Try training again")
# else:
#     print(f"\n OverflowError should be prevented now!")

print("="*60)

Checking train dataset...
 Train dataset token IDs look good
Checking dev dataset...
 Dev dataset token IDs look good
 Testing data collator safety...
   Collated input min: 1
   Collated label min: 1
 Data collator produces safe values
 Checking training arguments...

 OverflowError prevention complete!
No fixes needed - configuration looks good


In [13]:
# Create trainer (Seq2SeqTrainer with metrics) - FIXED FOR OVERFLOW ERROR
import numpy as np

# Use evaluate for metrics (modern replacement for datasets.load_metric)
try:
    from evaluate import load as load_metric
    sacrebleu = load_metric('sacrebleu')
    chrf = load_metric('chrf')
except Exception as _e:
    sacrebleu = None
    chrf = None
    print("Metrics load warning:", str(_e))

def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [l.strip() for l in labels]
    return preds, labels

# Compute exact-match, BLEU, chrF on decoded strings - FIXED FOR OVERFLOW ERROR
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # 🔧 FIX: Filter out negative token IDs that cause OverflowError
    print(f" Debug: Predictions shape: {preds.shape}")
    print(f" Debug: Predictions min/max: {preds.min()} / {preds.max()}")
    
    # Filter out negative predictions and replace with pad token
    preds_filtered = np.where(preds < 0, tokenizer.pad_token_id, preds)
    negative_count = np.sum(preds < 0)
    if negative_count > 0:
        print(f"  Fixed {negative_count} negative token IDs in predictions")
    
    # Decode with filtered predictions
    try:
        decoded_preds = tokenizer.batch_decode(preds_filtered, skip_special_tokens=True)
        print(f" Successfully decoded {len(decoded_preds)} predictions")
    except Exception as e:
        print(f"❌ Decoding predictions failed: {str(e)}")
        # Fallback: return dummy metrics
        return {
            'eval_exact_match': 0.0,
            'eval_bleu': 0.0,
            'eval_chrf': 0.0
        }

    # Replace -100 in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    # 🔧 FIX: Also filter negative labels
    labels_filtered = np.where(labels < 0, tokenizer.pad_token_id, labels)
    negative_label_count = np.sum(labels < 0)
    if negative_label_count > 0:
        print(f"  Fixed {negative_label_count} negative token IDs in labels")
    
    try:
        decoded_labels = tokenizer.batch_decode(labels_filtered, skip_special_tokens=True)
        print(f" Successfully decoded {len(decoded_labels)} labels")
    except Exception as e:
        print(f" Decoding labels failed: {str(e)}")
        # Fallback: return dummy metrics
        return {
            'eval_exact_match': 0.0,
            'eval_bleu': 0.0,
            'eval_chrf': 0.0
        }

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    # Exact match
    exact = np.mean([int(p == l) for p, l in zip(decoded_preds, decoded_labels)])

    result = {
        'exact_match': exact,
    }

    # BLEU/chrF if available
    if sacrebleu is not None:
        try:
            bleu_score = sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
            result['bleu'] = bleu_score.get('score', 0.0)
        except Exception as e:
            print(f"  BLEU computation failed: {str(e)}")
            result['bleu'] = 0.0
            
    if chrf is not None:
        try:
            chrf_score = chrf.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
            result['chrf'] = chrf_score.get('score', 0.0)
        except Exception as e:
            print(f"  chrF computation failed: {str(e)}")
            result['chrf'] = 0.0

    print(f" Metrics computed - Exact: {exact:.3f}, BLEU: {result.get('bleu', 0):.3f}, chrF: {result.get('chrf', 0):.3f}")

    # Trainer expects keys prefixed with eval_ during evaluation
    return {f"eval_{k}": v for k, v in result.items()}

# Pass generation parameters directly if needed
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=dev_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=CONFIG['early_stopping_patience'])
    ]
)

print(" Trainer created with OverflowError-safe compute_metrics function!")
print("   - Negative token IDs will be filtered before decoding")
print("   - Robust error handling for metric computation")
print("   - Debug information for troubleshooting")

Metrics load warning: To be able to use evaluate-metric/sacrebleu, you need to install the following dependencies['sacrebleu'] using 'pip install sacrebleu' for instance'
 Trainer created with OverflowError-safe compute_metrics function!
   - Negative token IDs will be filtered before decoding
   - Robust error handling for metric computation
   - Debug information for troubleshooting


In [ ]:
# Start training with enhanced checkpoint handling
print(f" Starting mT5 training with enhanced error handling...")
print(f"="*60)
print(f"Dataset: {len(train_tokenized):,} train + {len(dev_tokenized):,} dev samples")
print(f"Model: {CONFIG['model_name']}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Output: {training_args.output_dir}")
print(f"Resume from checkpoint: {CONFIG.get('resume_from_checkpoint', False)}")
if CONFIG.get('resume_checkpoint_path'):
    print(f"Resume checkpoint path: {CONFIG.get('resume_checkpoint_path')}")
print(f"="*60)

def safe_train_with_fallbacks():
    """Train with multiple fallback strategies for checkpoint issues"""
    
    # Clear any previous training state
    if CONFIG['device'] == 'cuda':
        torch.cuda.empty_cache()
    
    # Strategy 1: Try with configured checkpoint (if enabled)
    if CONFIG.get('resume_from_checkpoint', False):
        checkpoint_path = CONFIG.get('resume_checkpoint_path')
        
        if checkpoint_path and Path(checkpoint_path).exists():
            print(f" Strategy 1: Resuming from specific checkpoint: {checkpoint_path}")
            
            # Pre-flight check: verify batch size compatibility
            trainer_state_path = Path(checkpoint_path) / "trainer_state.json"
            if trainer_state_path.exists():
                try:
                    with open(trainer_state_path, 'r') as f:
                        state = json.load(f)
                    saved_batch_size = state.get('train_batch_size')
                    current_batch_size = training_args.per_device_train_batch_size
                    
                    if saved_batch_size and saved_batch_size != current_batch_size:
                        print(f"  Batch size mismatch detected: saved={saved_batch_size}, current={current_batch_size}")
                        print(f" Attempting to auto-fix...")
                        
                        # Try to fix the trainer state
                        state['train_batch_size'] = current_batch_size
                        backup_path = trainer_state_path.with_suffix('.json.backup')
                        
                        # Create backup and fix
                        import shutil
                        shutil.copy2(trainer_state_path, backup_path)
                        with open(trainer_state_path, 'w') as f:
                            json.dump(state, f, indent=2)
                        print(f" Fixed batch size and created backup at {backup_path}")
                        
                except Exception as fix_e:
                    print(f"  Could not auto-fix batch size: {str(fix_e)}")
            
            try:
                return trainer.train(resume_from_checkpoint=checkpoint_path)
            except Exception as e:
                error_msg = str(e).lower()
                if "missing keys" in error_msg or "embed_tokens" in error_msg:
                    print(f" Strategy 1 failed: Missing embedding keys")
                    print(f"   This is a known issue with checkpoint compatibility")
                elif "batch_size" in error_msg:
                    print(f" Strategy 1 failed: Batch size incompatibility")
                else:
                    print(f" Strategy 1 failed: {str(e)}")
                
                print(f" Trying fallback strategies...")
        
        # Strategy 2: Try latest checkpoint in output directory
        if Path(CONFIG['output_dir']).exists():
            checkpoints = list(Path(CONFIG['output_dir']).glob('checkpoint-*'))
            if checkpoints:
                latest_checkpoint = max(checkpoints, key=lambda x: int(x.name.split('-')[1]))
                print(f" Strategy 2: Trying latest checkpoint: {latest_checkpoint}")
                
                try:
                    return trainer.train(resume_from_checkpoint=str(latest_checkpoint))
                except Exception as e:
                    print(f" Strategy 2 failed: {str(e)}")
    
    # Strategy 3: Fresh training
    print(f" Strategy 3: Starting fresh training (no checkpoint)")
    try:
        return trainer.train()
    except Exception as e:
        print(f" All strategies failed!")
        raise e

try:
    print(f" Beginning training with fallback strategies...")
    training_output = safe_train_with_fallbacks()
    
    print(f"\n TRAINING COMPLETED SUCCESSFULLY!")
    print(f"Final training loss: {training_output.training_loss:.4f}")
    
    # Save final model
    print(f"\n Saving final model...")
    trainer.save_model()
    tokenizer.save_pretrained(CONFIG['output_dir'])
    
    # Save configuration with training completion timestamp
    config_to_save = CONFIG.copy()
    config_to_save['training_completed_at'] = str(pd.Timestamp.now())
    config_to_save['final_training_loss'] = float(training_output.training_loss)
    
    with open(f"{CONFIG['output_dir']}/config.json", 'w') as f:
        json.dump(config_to_save, f, indent=2, default=str)
    
    print(f"✅ Model saved to: {CONFIG['output_dir']}")
    
    # Final evaluation
    print(f"\n Final evaluation on dev set...")
    final_eval = trainer.evaluate()
    print(f"Final evaluation loss: {final_eval['eval_loss']:.4f}")
    print(f"Evaluation runtime: {final_eval['eval_runtime']:.1f} seconds")
    print(f"Samples per second: {final_eval['eval_samples_per_second']:.1f}")
    
    # Store results in globals
    globals()['trained_model'] = model
    globals()['trained_tokenizer'] = tokenizer
    globals()['training_results'] = training_output
    globals()['final_eval_results'] = final_eval
    globals()['training_completed'] = True
    
    print(f"\n🎉 Training pipeline completed successfully!")
    print(f"📈 Training metrics:")
    print(f"   • Final loss: {training_output.training_loss:.4f}")
    print(f"   • Eval loss: {final_eval['eval_loss']:.4f}")
    print(f"   • Model saved: {CONFIG['output_dir']}")
    
except Exception as e:
    print(f"\n💥 Training failed with error:")
    print(f"Error: {str(e)}")
    
    print(f"\n🔍 Debugging information:")
    print(f"   Model dtype: {model.dtype}")
    print(f"   Training args fp16: {training_args.fp16}")
    print(f"   Batch size (config): {CONFIG['train_batch_size']}")
    print(f"   Batch size (args): {training_args.per_device_train_batch_size}")
    
    # Show memory info if CUDA
    if CONFIG['device'] == 'cuda':
        print(f"\n🖥️  GPU Memory Status:")
        print(f"   Allocated: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
        print(f"   Reserved: {torch.cuda.memory_reserved() / 1024**3:.1f} GB")
    
    # Enhanced checkpoint debugging
    if CONFIG.get('resume_from_checkpoint') and CONFIG.get('resume_checkpoint_path'):
        checkpoint_path = Path(CONFIG['resume_checkpoint_path'])
        print(f"\n🔍 Checkpoint debugging:")
        print(f"   Checkpoint path exists: {checkpoint_path.exists()}")
        if checkpoint_path.exists():
            print(f"   Checkpoint contents: {list(checkpoint_path.iterdir())}")
            
            # Check for specific issues
            trainer_state_path = checkpoint_path / "trainer_state.json"
            if trainer_state_path.exists():
                try:
                    with open(trainer_state_path, 'r') as f:
                        state = json.load(f)
                    print(f"   Saved batch size: {state.get('train_batch_size')}")
                    print(f"   Current batch size: {CONFIG['train_batch_size']}")
                except:
                    print(f"   Could not read trainer_state.json")
    
    print(f"\n💡 Troubleshooting suggestions:")
    print(f"   1. Run the checkpoint recovery tools (previous cell)")
    print(f"   2. Set 'resume_from_checkpoint': False to start fresh")
    print(f"   3. Check that all checkpoint files are present and valid")
    print(f"   4. Verify that batch sizes match between config and checkpoint")
    
    raise

 Starting mT5 training with enhanced error handling...
Dataset: 13,253 train + 2,752 dev samples
Model: google/mt5-small
Effective batch size: 32
Learning rate: 0.0001
Epochs: 3
Output: ./mt5-hindi-final-v4
Resume from checkpoint: True
Resume checkpoint path: ./mt5-hindi-final-v3/checkpoint-1000
 Beginning training with fallback strategies...
 Strategy 2: Trying latest checkpoint: mt5-hindi-final-v4\checkpoint-100


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Step,Training Loss,Validation Loss,Exact Match
200,1.312600,0.626199,0.000000


 Debug: Predictions shape: (2752, 128)
 Debug: Predictions min/max: -100 / 250099
  Fixed 175920 negative token IDs in predictions
 Successfully decoded 2752 predictions
 Successfully decoded 2752 labels
 Metrics computed - Exact: 0.000, BLEU: 0.000, chrF: 0.000


In [17]:
# Test the trained mT5 model
print(f"Testing the trained mT5 model...")
print(f"="*50)

import re

# Helper: collect sentinel token ids (<extra_id_0..99>) to block during generation
_def_sentinel_cache = {}

def _get_sentinel_bad_word_ids(tok, max_sentinels: int = 100):
    global _def_sentinel_cache
    if id(tok) in _def_sentinel_cache:
        return _def_sentinel_cache[id(tok)]
    ids = []
    # Prefer tokenizer.additional_special_tokens when available
    specials = getattr(tok, 'additional_special_tokens', []) or []
    for t in specials:
        if isinstance(t, str) and t.startswith("<extra_id_") and t.endswith(">"):
            tid = tok.convert_tokens_to_ids(t)
            if tid is not None and tid != tok.unk_token_id:
                ids.append([tid])
    # Fallback: probe names
    if not ids:
        for i in range(max_sentinels):
            t = f"<extra_id_{i}>"
            tid = tok.convert_tokens_to_ids(t)
            if tid is not None and tid != tok.unk_token_id:
                ids.append([tid])
    _def_sentinel_cache[id(tok)] = ids
    return ids

# Test sentences with various Hindi grammar errors
test_sentences = [
    "मैं कल दिल्ली जाऊगा",           # Missing anusvara (should be जाऊंगा)
    "वो स्कूल गया हैं",              # Verb agreement (should be गया है)
    "राम और श्याम खेल रहा है",         # Plural subject (should be खेल रहे हैं)
    "मुझे यह किताब पसंद हैं",         # Agreement error (should be पसंद है)
    "बच्चे पार्क में खेल रहे हैं",       # Correct sentence
    "उसके पास बहुत पैसा हैं",         # Agreement error (should be पैसा है)
    "हम सब मिलकर काम करेगे",         # Spelling/verb (should be करेंगे)
    "तुम कहा जा रहे हो",            # Interrogative (should be कहाँ)
    "भारत का पहला स्वदेशी टीका",
]

print(f"Test Results:")
print(f"-" * 80)

# Helper to get a ready model/tokenizer (use in-memory if available; else load from disk)
def _get_active_model_and_tokenizer():
    active_model = None
    active_tokenizer = None

    # Prefer in-memory trained artifacts
    if 'model' in globals() and 'tokenizer' in globals():
        active_model = globals()['model']
        active_tokenizer = globals()['tokenizer']
    elif 'trained_model' in globals() and 'trained_tokenizer' in globals():
        active_model = globals()['trained_model']
        active_tokenizer = globals()['trained_tokenizer']

    # Otherwise, load from checkpoint directory
    if active_model is None or active_tokenizer is None:
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        print(f"Loading trained artifacts from: {CONFIG['output_dir']}")
        active_tokenizer = AutoTokenizer.from_pretrained(CONFIG['output_dir'])
        active_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG['output_dir'], torch_dtype=torch.float32)

    # Device and eval mode
    device = CONFIG.get('device', 'cuda' if torch.cuda.is_available() else 'cpu')
    active_model = active_model.to(device)
    active_model.eval()

    # Safety: ensure pad token id exists and decoder start token is set
    if active_tokenizer.pad_token_id is None and active_tokenizer.eos_token_id is not None:
        active_tokenizer.pad_token = active_tokenizer.eos_token
    if getattr(active_model.config, 'decoder_start_token_id', None) is None and active_tokenizer.pad_token_id is not None:
        active_model.config.decoder_start_token_id = active_tokenizer.pad_token_id

    # Debug: show which checkpoint we are using
    print(f"Using checkpoint: {getattr(active_model.config, '_name_or_path', 'unknown')}")

    return active_model, active_tokenizer


def correct_hindi_grammar_t5(text, max_length=None):
    """Correct Hindi grammar using the trained mT5 model (uses the same prefix as training)."""
    active_model, active_tokenizer = _get_active_model_and_tokenizer()

    gen_cfg = CONFIG.get('generation_config', {})
    if max_length is None:
        max_length = gen_cfg.get('max_new_tokens', 128)

    # Use the same task prefix as in training/tokenization
    input_text = f"grammar: {text.strip()}"
    inputs = active_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=CONFIG['max_length'],
        truncation=True,
        padding=True
    )
    inputs = {k: v.to(active_model.device) for k, v in inputs.items()}

    # Build bad words list to block sentinel tokens (<extra_id_#>)
    bad_words_ids = _get_sentinel_bad_word_ids(active_tokenizer)

    with torch.no_grad():
        outputs = active_model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_length,
            num_beams=gen_cfg.get('num_beams', 4),
            do_sample=gen_cfg.get('do_sample', False),
            repetition_penalty=gen_cfg.get('repetition_penalty', 1.2),
            length_penalty=gen_cfg.get('length_penalty', 1.0),
            early_stopping=gen_cfg.get('early_stopping', True),
            eos_token_id=active_tokenizer.eos_token_id,
            pad_token_id=active_tokenizer.pad_token_id,
            no_repeat_ngram_size=3,
            bad_words_ids=bad_words_ids if bad_words_ids else None,
        )
    result = active_tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Final cleanup: strip any stray sentinel tokens if tokenizer didn't mark them as special
    result = re.sub(r"<extra_id_\d+>", "", result).strip()
    return result if result else text

for i, sentence in enumerate(test_sentences, 1):
    try:
        corrected = correct_hindi_grammar_t5(sentence)
        
        # Determine if changed
        changed = sentence.strip() != corrected.strip()
        status_icon = "✓ CORRECTED" if changed else "○ UNCHANGED"
        
        print(f"\n{i:2d}. {status_icon}")
        print(f"     Original:  {sentence}")
        print(f"     Corrected: {corrected}")
        
        # If changed, show what was corrected
        if changed:
            print(f"     -> Grammar correction applied")
        
    except Exception as e:
        print(f"\n{i:2d}. ✗ ERROR")
        print(f"     Original: {sentence}")
        print(f"     Error: {str(e)[:60]}...")

print(f"\n" + "="*50)
print(f"mT5 model testing completed!")

Testing the trained mT5 model...
Test Results:
--------------------------------------------------------------------------------
Using checkpoint: google/mt5-small

 1. ○ UNCHANGED
     Original:  मैं कल दिल्ली जाऊगा
     Corrected: मैं कल दिल्ली जाऊगा
Using checkpoint: google/mt5-small

 2. ○ UNCHANGED
     Original:  वो स्कूल गया हैं
     Corrected: वो स्कूल गया हैं
Using checkpoint: google/mt5-small

 3. ○ UNCHANGED
     Original:  राम और श्याम खेल रहा है
     Corrected: राम और श्याम खेल रहा है
Using checkpoint: google/mt5-small

 4. ○ UNCHANGED
     Original:  मुझे यह किताब पसंद हैं
     Corrected: मुझे यह किताब पसंद हैं
Using checkpoint: google/mt5-small

 1. ○ UNCHANGED
     Original:  मैं कल दिल्ली जाऊगा
     Corrected: मैं कल दिल्ली जाऊगा
Using checkpoint: google/mt5-small

 2. ○ UNCHANGED
     Original:  वो स्कूल गया हैं
     Corrected: वो स्कूल गया हैं
Using checkpoint: google/mt5-small

 3. ○ UNCHANGED
     Original:  राम और श्याम खेल रहा है
     Corrected: राम और श्याम खेल र

In [ ]:
# Load the trained model from saved checkpoint (optional helper)
print("LOADING TRAINED MODEL (helper cell):")
print("=" * 40)

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# Reuse the global CONFIG defined earlier to avoid mismatched paths
print(f"Using output_dir from CONFIG: {CONFIG['output_dir']}")

try:
    # Load tokenizer and model from training output_dir
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['output_dir'])
    print(f"✓ Tokenizer loaded: {type(tokenizer).__name__}")

    model = AutoModelForSeq2SeqLM.from_pretrained(
        CONFIG['output_dir'],
        torch_dtype=torch.float32
    )
    print(f"✓ Model loaded: {type(model).__name__}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

    # Move to device and eval mode
    model = model.to(CONFIG['device'])
    model.eval()
    print(f"✓ Model moved to: {CONFIG['device']}")
    print(f"✓ Model set to evaluation mode")

    print("✓ Trained model loaded successfully!")

except Exception as e:
    print(f"✗ Error loading model: {str(e)}")
    print("You may need to run the training cells first")

print("=" * 40)

LOADING TRAINED MODEL:


d:\CODING\IndicGEC2025\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model from: ./mt5-hindi-final


`torch_dtype` is deprecated! Use `dtype` instead!


✓ Tokenizer loaded: T5TokenizerFast
✓ Model loaded: MT5ForConditionalGeneration
  Parameters: 300.2M
✓ Model moved to: cuda
✓ Model set to evaluation mode
✓ Trained model loaded successfully!


In [25]:
# Pre-Training Safety Test
print(" Pre-Training Safety Test")
print("="*50)

def pre_training_safety_test():
    """Run comprehensive safety checks before training"""
    
    print(" Running safety checks...")
    
    # Test 1: Tokenization safety
    test_inputs = [
        "इस वाक्य में कोई गलती है।",
        "मैं स्कूल जा रहा हूं।",
        "वह अच्छा छात्र है।"
    ]
    
    test_targets = [
        "इस वाक्य में कोई गलती है।",
        "मैं स्कूल जा रहा हूं।", 
        "वह एक अच्छा छात्र है।"
    ]
    
    print("1️ Testing tokenization safety...")
    try:
        for i, (inp, tgt) in enumerate(zip(test_inputs, test_targets)):
            result = tokenize_function({
                'input_text': [inp],  # Fixed: using correct key names
                'output_text': [tgt]  # Fixed: using correct key names
            })
            
            input_ids = result['input_ids'][0]
            labels = result['labels'][0]
            
            # Check for negative values
            if any(id < 0 for id in input_ids):
                print(f" Negative input_ids in test {i+1}")
                return False
                
            if any(id < 0 for id in labels if id != -100):  # -100 is allowed for padding
                print(f" Negative labels in test {i+1}")
                return False
                
            print(f"    Test {i+1} passed")
            print(f"   All tokenization tests passed!")

    except Exception as e:
        print(f" Tokenization test failed: {str(e)}")
        return False
    
    # Test 2: Data collator safety
    print("2️ Testing data collator safety...")
    try:
        if 'train_tokenized' in globals():
            test_batch = [train_tokenized[i] for i in range(min(3, len(train_tokenized)))]
            collated = data_collator(test_batch)
            
            # Check shapes and values
            batch_size = collated['input_ids'].shape[0]
            seq_len = collated['input_ids'].shape[1]
            
            min_input = collated['input_ids'].min().item()
            max_input = collated['input_ids'].max().item()
            min_label = collated['labels'].min().item()
            max_label = collated['labels'].max().item()
            
            print(f"   Batch shape: {batch_size} x {seq_len}")
            print(f"   Input range: {min_input} to {max_input}")
            print(f"   Label range: {min_label} to {max_label}")
            
            # Sanity checks
            if min_input < 0 or max_input >= len(tokenizer):
                print(f" Input IDs out of valid range!")
                return False
                
            if min_label < -100 or (max_label >= len(tokenizer) and max_label != -100):
                print(f" Label IDs out of valid range!")
                return False
                
            print("    Data collator test passed!")
        else:
            print("    No train_tokenized dataset to test")
            
    except Exception as e:
        print(f" Data collator test failed: {str(e)}")
        return False
    
    # Test 3: Model forward pass safety
    print("3️ Testing model forward pass...")
    try:
        if 'model' in globals() and 'train_tokenized' in globals():
            model.eval()
            
            # Get a small batch
            test_batch = [train_tokenized[0]]
            batch = data_collator(test_batch)
            
            # Move to device
            device = next(model.parameters()).device
            batch = {k: v.to(device) if hasattr(v, 'to') else v for k, v in batch.items()}
            
            # Forward pass
            with torch.no_grad():
                outputs = model(**batch)
                loss = outputs.loss
                
            print(f"   Forward pass successful!")
            print(f"   Loss: {loss.item():.4f}")
            print("   Model forward pass test passed!")
            
        else:
            print("   No model or dataset to test")
            
    except Exception as e:
        print(f" Model forward pass failed: {str(e)}")
        return False
    
    # Test 4: Training args validation
    print("4️Validating training arguments...")
    if 'training_args' in globals():
        print(f"   Batch size: {training_args.per_device_train_batch_size}")
        print(f"   Learning rate: {training_args.learning_rate}")
        print(f"   Epochs: {training_args.num_train_epochs}")
        print(f"   Output dir: {training_args.output_dir}")
        print("   Training arguments look good!")
    else:
        print("   No training arguments defined")
    
    print("\n All safety tests passed! Training should be safe to start.")
    return True

# Run the safety test
print("Starting comprehensive safety test...")
safety_passed = pre_training_safety_test()

if safety_passed:
    print("\n Ready for training!")
    print("You can now run the training cell safely.")
else:
    print("\n Safety test failed!")
    print("Please fix the issues before training.")

print("="*50)

🧪 Pre-Training Safety Test
Starting comprehensive safety test...
🔍 Running safety checks...
1️⃣ Testing tokenization safety...
   ✅ Test 1 passed
   ✅ Test 2 passed
   ✅ Test 3 passed
   ✅ All tokenization tests passed!
2️⃣ Testing data collator safety...
   Batch shape: 3 x 56
   Input range: 0 to 218380
   Label range: -100 to 178009
   ✅ Data collator test passed!
3️⃣ Testing model forward pass...
   Forward pass successful!
   Loss: 37.6243
   ✅ Model forward pass test passed!
4️⃣ Validating training arguments...
   Batch size: 1
   Learning rate: 0.0003
   Epochs: 2
   Output dir: ./mt5-hindi-gec-emergency
   ✅ Training arguments look good!

🎉 All safety tests passed! Training should be safe to start.

✅ Ready for training!
You can now run the training cell safely.
   Forward pass successful!
   Loss: 37.6243
   ✅ Model forward pass test passed!
4️⃣ Validating training arguments...
   Batch size: 1
   Learning rate: 0.0003
   Epochs: 2
   Output dir: ./mt5-hindi-gec-emergency
   ✅